In [25]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from scipy.stats import ks_2samp

# =========================================================
# CONFIG
# =========================================================
RANK = 1

TARGET_COL = "status_fraude"

THRESHOLD = 0.50

NOME_HTML = f"2d_rank_{RANK}_orig.html"

# =========================================================
# DIRETÓRIO
# =========================================================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

print("Diretório:", BASE_DIR)

# =========================================================
# LOAD RANKING
# =========================================================
df_scores = pd.read_csv("2x2_visu_scores.csv")

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features: {feature_1} vs {feature_2}")

# =========================================================
# LOAD DATASET
# =========================================================
df = pd.read_csv("creditcard.csv")

# =========================================================
# DATASET ORIGINAL COMPLETO
# =========================================================
df_model = df[
    [feature_1, feature_2, TARGET_COL]
].dropna()

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# =========================================================
# FEATURES
# =========================================================
X = df_model[[feature_1, feature_2]]
y = df_model[TARGET_COL]

# =========================================================
# SCALE
# =========================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================================================
# GMM
# =========================================================
gmm = GaussianMixture(
    n_components=2,
    covariance_type="full",
    random_state=42,
    reg_covar=1e-6,
    n_init=3
)

gmm.fit(X_scaled)

# =========================================================
# CLUSTERS
# =========================================================
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:
    raise ValueError("Nenhuma fraude encontrada nos clusters.")

cluster_fraude = ct[1].idxmax()

print(f"\nCluster identificado como fraude: {cluster_fraude}")

# =========================================================
# PROBABILIDADES
# =========================================================
score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# =========================================================
# PREDIÇÃO
# =========================================================
y_pred = (score >= THRESHOLD).astype(int)

# =========================================================
# MÉTRICAS
# =========================================================
prec, rec, _ = precision_recall_curve(y, score)

auc_pr = auc(rec, prec)

f1 = f1_score(y, y_pred)

mcc = matthews_corrcoef(y, y_pred)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(y, score)

# =========================================================
# MATRIZ CONFUSÃO
# =========================================================
cm = confusion_matrix(y, y_pred)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):

    linha = []

    for j in range(2):

        linha.append(
            f"{cm_percent[i, j]:.2f}%<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# =========================================================
# MATRIZ CORRELAÇÃO SPEARMAN
# =========================================================
corr = df_model[
    [feature_1, feature_2]
].corr(method="spearman")

# =========================================================
# DATASETS SCATTER
# =========================================================
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# =========================================================
# LIMITES DO SCATTER COM FOLGA
# =========================================================
x_min = df_model[feature_1].min()
x_max = df_model[feature_1].max()

y_min = df_model[feature_2].min()
y_max = df_model[feature_2].max()

x_pad = (x_max - x_min) * 0.08
y_pad = (y_max - y_min) * 0.08

x_range = [
    x_min - x_pad,
    x_max + x_pad
]

y_range = [
    y_min - y_pad,
    y_max + y_pad
]

# =========================================================
# FIGURA
# =========================================================
fig = make_subplots(

    rows=3,
    cols=2,

    specs=[

        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],

        [
            {"colspan": 2},
            None
        ],

        [
            {"colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.28,
        0.16,
        0.56
    ],

    horizontal_spacing=0.12,
    vertical_spacing=0.13,

    subplot_titles=(
        "Correlação Spearman",
        "Matriz de Confusão (%)",
        "",
        "Distribuição 2D das Features"
    )
)

# =========================================================
# MATRIZ CORRELAÇÃO
# =========================================================
fig.add_trace(

    go.Heatmap(

        z=corr.values,

        x=[
            feature_1,
            feature_2
        ],

        y=[
            feature_1,
            feature_2
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# =========================================================
# MATRIZ CONFUSÃO
# =========================================================
fig.add_trace(

    go.Heatmap(

        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# =========================================================
# MÉTRICAS
# =========================================================
metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
F1 Score: {f1:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}
"""

fig.add_trace(

    go.Scatter(

        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=18
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# =========================================================
# SCATTER NÃO FRAUDE
# =========================================================
fig.add_trace(

    go.Scattergl(

        x=df_nao_fraude[feature_1],

        y=df_nao_fraude[feature_2],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.25)",
            size=5
        )
    ),

    row=3,
    col=1
)

# =========================================================
# SCATTER FRAUDE
# =========================================================
fig.add_trace(

    go.Scattergl(

        x=df_fraude[feature_1],

        y=df_fraude[feature_2],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.90)",
            size=5
        )
    ),

    row=3,
    col=1
)

# =========================================================
# AXIS
# =========================================================
fig.update_xaxes(
    title_text=feature_1,
    range=x_range,
    row=3,
    col=1
)

fig.update_yaxes(
    title_text=feature_2,
    range=y_range,
    row=3,
    col=1
)

# =========================================================
# LAYOUT
# =========================================================
fig.update_layout(

    title=dict(

        text=f"""
        Relatório GMM
        <br>
        Rank {RANK}
        <br>
        {feature_1} vs {feature_2}
        <br>
        Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
        """,

        x=0.5,
        y=0.98,

        xanchor="center",
        yanchor="top"
    ),

    width=1850,
    height=1900,

    template="plotly_white",

    font=dict(
        size=16
    ),

    margin=dict(
        t=270,
        b=130,
        l=90,
        r=90
    ),

    legend=dict(

        orientation="h",

        yanchor="bottom",
        y=0.01,

        xanchor="center",
        x=0.5
    )
)

# =========================================================
# SAVE HTML
# =========================================================
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V15 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              280486  115
1                3829  377

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\2d_rank_1_orig.html


#### er

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import time

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from scipy.stats import ks_2samp
from openTSNE import TSNE

# =========================================================
# CONFIG
# =========================================================
RANK = 1

TARGET_COL = "status_fraude"

THRESHOLD = 0.50

NOME_HTML = f"2d_rank_{RANK}_tsne.html"

# =========================================================
# HIPERPARÂMETROS t-SNE
# =========================================================
TSNE_PERPLEXITY = 30

TSNE_N_ITER = 300

TSNE_EARLY_EXAGGERATION_ITER = 100

TSNE_EARLY_EXAGGERATION = 12

TSNE_EXAGGERATION = 1

TSNE_LEARNING_RATE = "auto"

TSNE_METRIC = "euclidean"

TSNE_INITIALIZATION = "pca"

TSNE_NEGATIVE_GRADIENT_METHOD = "bh"

TSNE_N_JOBS = -1

TSNE_RANDOM_STATE = 42

TSNE_VERBOSE = True

# =========================================================
# DIRETÓRIO
# =========================================================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

HTML_PATH = os.path.join(
    BASE_DIR,
    NOME_HTML
)

print("Diretório:", BASE_DIR)

# =========================================================
# LOAD RANKING
# =========================================================
df_scores = pd.read_csv(
    "2x2_visu_scores.csv"
)

row = df_scores[
    df_scores["Posicao_Rank"] == RANK
].iloc[0]

feature_1 = row["Feature_1"]
feature_2 = row["Feature_2"]

print(f"\nRank Selecionado: {RANK}")
print(f"Features originais: {feature_1} vs {feature_2}")

# =========================================================
# LOAD DATASET
# =========================================================
df = pd.read_csv("creditcard.csv")

# =========================================================
# DATASET ORIGINAL COMPLETO
# =========================================================
df_model = df[
    [feature_1, feature_2, TARGET_COL]
].dropna().reset_index(drop=True)

print("\nQuantidade usada:")
print(df_model[TARGET_COL].value_counts())

# =========================================================
# FEATURES ORIGINAIS
# =========================================================
X_original = df_model[
    [feature_1, feature_2]
]

y = df_model[
    TARGET_COL
]

# =========================================================
# SCALE ANTES DO t-SNE
# =========================================================
scaler_original = StandardScaler()

X_scaled_original = scaler_original.fit_transform(
    X_original
)

# =========================================================
# t-SNE
# =========================================================
print("\nRodando t-SNE...\n")

inicio_tsne = time.perf_counter()

tsne = TSNE(

    n_components=2,

    perplexity=TSNE_PERPLEXITY,

    learning_rate=TSNE_LEARNING_RATE,

    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,

    early_exaggeration=TSNE_EARLY_EXAGGERATION,

    n_iter=TSNE_N_ITER,

    exaggeration=TSNE_EXAGGERATION,

    metric=TSNE_METRIC,

    initialization=TSNE_INITIALIZATION,

    negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,

    n_jobs=TSNE_N_JOBS,

    random_state=TSNE_RANDOM_STATE,

    verbose=TSNE_VERBOSE
)

X_tsne = tsne.fit(X_scaled_original)

X_tsne = np.asarray(X_tsne)

fim_tsne = time.perf_counter()

print(
    f"\nt-SNE finalizado em "
    f"{(fim_tsne - inicio_tsne):.2f} segundos."
)

# =========================================================
# NOVAS FEATURES t-SNE
# =========================================================
df_model["TSNE_1"] = X_tsne[:, 0]

df_model["TSNE_2"] = X_tsne[:, 1]

feature_tsne_1 = "TSNE_1"
feature_tsne_2 = "TSNE_2"

# =========================================================
# FEATURES PARA O GMM
# =========================================================
X = df_model[
    [feature_tsne_1, feature_tsne_2]
]

# =========================================================
# SCALE APÓS t-SNE
# =========================================================
scaler_gmm = StandardScaler()

X_scaled = scaler_gmm.fit_transform(X)

# =========================================================
# GMM
# =========================================================
gmm = GaussianMixture(

    n_components=2,

    covariance_type="full",

    random_state=42,

    reg_covar=1e-6,

    n_init=3
)

gmm.fit(X_scaled)

# =========================================================
# CLUSTERS
# =========================================================
clusters = gmm.predict(X_scaled)

ct = pd.crosstab(
    clusters,
    y
)

print("\nTabela Cluster x Classe Real:")
print(ct)

if 1 not in ct.columns:

    raise ValueError(
        "Nenhuma fraude encontrada nos clusters."
    )

cluster_fraude = ct[1].idxmax()

print(
    f"\nCluster identificado como fraude: "
    f"{cluster_fraude}"
)

# =========================================================
# PROBABILIDADES
# =========================================================
score = gmm.predict_proba(
    X_scaled
)[:, cluster_fraude]

score = np.clip(
    score,
    1e-15,
    1 - 1e-15
)

# =========================================================
# PREDIÇÃO
# =========================================================
y_pred = (
    score >= THRESHOLD
).astype(int)

# =========================================================
# MÉTRICAS
# =========================================================
prec, rec, _ = precision_recall_curve(
    y,
    score
)

auc_pr = auc(
    rec,
    prec
)

f1 = f1_score(
    y,
    y_pred
)

mcc = matthews_corrcoef(
    y,
    y_pred
)

ks = ks_2samp(
    score[y == 0],
    score[y == 1]
).statistic

ll = log_loss(
    y,
    score
)

# =========================================================
# MATRIZ CONFUSÃO
# =========================================================
cm = confusion_matrix(
    y,
    y_pred
)

cm_percent = (
    cm.astype(float)
    / cm.sum(axis=1)[:, np.newaxis]
) * 100

texto_cm = []

for i in range(2):

    linha = []

    for j in range(2):

        linha.append(
            f"{cm_percent[i, j]:.2f}%"
            f"<br>({cm[i, j]})"
        )

    texto_cm.append(linha)

# =========================================================
# MATRIZ CORRELAÇÃO SPEARMAN
# =========================================================
corr = df_model[
    ["TSNE_1", "TSNE_2"]
].corr(method="spearman")

# =========================================================
# DATASETS SCATTER
# =========================================================
df_fraude = df_model[
    df_model[TARGET_COL] == 1
]

df_nao_fraude = df_model[
    df_model[TARGET_COL] == 0
]

# =========================================================
# LIMITES DO SCATTER COM FOLGA
# =========================================================
x_min = df_model["TSNE_1"].min()
x_max = df_model["TSNE_1"].max()

y_min = df_model["TSNE_2"].min()
y_max = df_model["TSNE_2"].max()

x_pad = (x_max - x_min) * 0.15
y_pad = (y_max - y_min) * 0.15

x_range = [
    x_min - x_pad,
    x_max + x_pad
]

y_range = [
    y_min - y_pad,
    y_max + y_pad
]

# =========================================================
# FIGURA
# =========================================================
fig = make_subplots(

    rows=3,
    cols=2,

    specs=[

        [
            {"type": "heatmap"},
            {"type": "heatmap"}
        ],

        [
            {"colspan": 2},
            None
        ],

        [
            {"colspan": 2},
            None
        ]
    ],

    row_heights=[
        0.26,
        0.15,
        0.59
    ],

    horizontal_spacing=0.10,
    vertical_spacing=0.09,

    subplot_titles=(

        "Correlação Spearman - t-SNE",

        "Matriz de Confusão (%)",

        "",

        "Distribuição 2D após t-SNE"
    )
)

# =========================================================
# MATRIZ CORRELAÇÃO
# =========================================================
fig.add_trace(

    go.Heatmap(

        z=corr.values,

        x=[
            "TSNE_1",
            "TSNE_2"
        ],

        y=[
            "TSNE_1",
            "TSNE_2"
        ],

        text=np.round(
            corr.values,
            3
        ),

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="RdBu",

        zmin=-1,
        zmax=1,

        showscale=False
    ),

    row=1,
    col=1
)

# =========================================================
# MATRIZ CONFUSÃO
# =========================================================
fig.add_trace(

    go.Heatmap(

        z=cm_percent,

        x=[
            "Pred Não Fraude",
            "Pred Fraude"
        ],

        y=[
            "Real Não Fraude",
            "Real Fraude"
        ],

        text=texto_cm,

        texttemplate="%{text}",

        textfont=dict(
            size=18
        ),

        colorscale="Blues",

        zmin=0,
        zmax=100,

        showscale=False
    ),

    row=1,
    col=2
)

# =========================================================
# MÉTRICAS
# =========================================================
metricas = f"""
<b>MÉTRICAS</b><br><br>

AUC-PR: {auc_pr:.4f}<br>

F1 Score: {f1:.4f}<br>

MCC: {mcc:.4f}<br>

KS: {ks:.4f}<br>

Log Loss: {ll:.4f}
"""

fig.add_trace(

    go.Scatter(

        x=[0.5],
        y=[0.5],

        mode="text",

        text=[metricas],

        textfont=dict(
            size=20
        ),

        showlegend=False
    ),

    row=2,
    col=1
)

# =========================================================
# REMOVE EIXOS MÉTRICAS
# =========================================================
fig.update_xaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

fig.update_yaxes(
    visible=False,
    range=[0, 1],
    row=2,
    col=1
)

# =========================================================
# SCATTER NÃO FRAUDE
# =========================================================
fig.add_trace(

    go.Scattergl(

        x=df_nao_fraude["TSNE_1"],

        y=df_nao_fraude["TSNE_2"],

        mode="markers",

        name="Não Fraude",

        marker=dict(
            color="rgba(0,0,255,0.20)",
            size=5
        )
    ),

    row=3,
    col=1
)

# =========================================================
# SCATTER FRAUDE
# =========================================================
fig.add_trace(

    go.Scattergl(

        x=df_fraude["TSNE_1"],

        y=df_fraude["TSNE_2"],

        mode="markers",

        name="Fraude",

        marker=dict(
            color="rgba(255,0,0,0.95)",
            size=5
        )
    ),

    row=3,
    col=1
)

# =========================================================
# AXIS
# =========================================================
fig.update_xaxes(

    title_text="TSNE_1",

    title_font=dict(size=24),

    tickfont=dict(size=16),

    range=x_range,

    row=3,
    col=1
)

fig.update_yaxes(

    title_text="TSNE_2",

    title_font=dict(size=24),

    tickfont=dict(size=16),

    range=y_range,

    row=3,
    col=1
)

# =========================================================
# LAYOUT
# =========================================================
fig.update_layout(

    title=dict(

        text=f"""
        Relatório GMM após t-SNE (Spearman)
        <br>
        Rank {RANK}
        <br>
        Features originais:
        {feature_1} vs {feature_2}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2
        """,

        x=0.5,
        y=0.985,

        xanchor="center",
        yanchor="top",

        font=dict(size=28)
    ),

    width=1900,

    height=2150,

    template="plotly_white",

    font=dict(
        size=18
    ),

    margin=dict(
        t=320,
        b=170,
        l=120,
        r=120
    ),

    legend=dict(

        orientation="h",

        font=dict(size=18),

        yanchor="bottom",
        y=0.01,

        xanchor="center",
        x=0.5
    )
)

# =========================================================
# SAVE HTML
# =========================================================
fig.write_html(
    HTML_PATH,
    include_plotlyjs="cdn"
)

print("\nHTML GERADO COM SUCESSO:")
print(HTML_PATH)
#alterado